# Eager vs. Lazy Evaluation
## Eager Evaluation
![image](../notebooks//images/08/Eagerness.png)
- bedeutet: Jede Operation wird vollständig auf alle Elemente nacheinander angewendet

## Lazy Evaluation
![image](../notebooks/images/08/laziness.png)
- Operationen wie `.map()`, `.filter()`, `.findFirst()` werden verkettet und pro Element nacheinander angewendet  
=> spart Rechenzeit, weil unnötige Operationen auf spätere Elemente vermieden werden (*siehe Elemente 4–6*)

***Short-Circuit-Operation*** 
- ist eine Terminal- oder Intermediate-Operation in einem
Java-Stream, die nicht den gesamten Stream verarbeiten muss
- bei Erreichen eines bestimmten Ergebnisse kann die Ausführung frühzeitig beendet
- Bspw. findFirst, findAny, anyMatch

## Mutability

- Mutable Collection: Werte innerhalb der Collection sind veränderbar
- Immutable Collection: Werte innerhalb der Collection sind nicht veränderbar
Die Veränderlichkeit bezieht sich auf den Inhalt, **nicht** auf die Referenz

```scala
val l1 = List(1, 2, 3, 4, 5, 7)               // Immutable
val l2 = ListBuffer(1, 2, 3, 4, 5, 7)         // Mutable

l1 :+ 8        // gibt neue Liste zurück, l1 bleibt unverändert
l2.addOne(8)   // verändert l2 direkt
```

=> `List` ist immutable – alle Änderungen erzeugen neue Collections  
=> `ListBuffer` ist mutable – Änderungen wirken direkt auf die Datenstruktur

### Scala Collections lazy machen

Zwei Möglichkeiten, in Scala lazy zu arbeiten:

1. **LazyList**
2. **View**

#### LazyList

- Eigene Datenstruktur, funktioniert wie `List`, ist aber lazy
- Ergebnisse werden zwischengespeichert (***Caching***) und nur bei Bedarf berechnet

#### View
- Kein neuer Datentyp, sondern eine "lazy Sicht" auf eine bestehende Collection
- Transformationen werden erst beim Zugriff ausgeführt
- Kein Caching – jede Abfrage wird neu berechnet


=> view erlaubt Lazy Evaluation ohne neue Struktur

In [ ]:
val l1 = List(1, 2, 3, 4, 5, 7)
val lazy1 = l1.to(LazyList).take(3)
println(lazy1.toList)
// Ausgabe: List(1, 2, 3)

val view1 = l1.view.take(3)
println(view1.toList)
// Ausgabe: List(1, 2, 3)

List(1, 2, 3)


l1: List[Int] = List(1, 2, 3, 4, 5, 7)
lazy1: LazyList[Int] = LazyList(1, 2, 3)


## Monaden

Eine **Monade** ist ein *Wert mit Kontext*. Drei zentrale Beispiele in Scala:

- `Option[T]` : *Vielleicht ist da ein Wert, vielleicht nicht*
- `List[T]` : *Da könnten mehrere Werte drinne sein*
- `Future[T]` : *Ein Wert, der später verfügbar sein wird*



Rein formal braucht es 2 Dinge um eine Monade zu definieren:
- einem Konstruktor, der einen Wert in einen Kontext einpackt A => M[A].
- einer Methode flatMap, um Operationen im Kontext zu verketten. flatMap: M[A] => (A => M[B]) => M[B]
- Flatmap sollte linker Identität, rechter Identität und Assoziativität genügen (siehe Vorlesung 5)


flatMap erhält also eine Funktion und nutzt diese intern um wieder einen Wert mit Kontext zu berechnen (Monade).
- es abstrahiert das Sequenzieren von Berechnungen im Kontext und verhindert verschachtelte Strukturen wie M[M[B]]
- `map` erhält den doppelten Kontext: `List[List[Int]]`
- `flatMap` entfernt den inneren Kontext: `List[Int]`

flatMap nimmt einen Wert im Kontext, wendet eine Funktion welche den Wert ohne Kontext bekommt und wieder einen Kontext zurückgibt an, und sorgt dafür, dass kein verschachtelter Kontext entsteht.

In [1]:
def half(x: Int): List[Int] =
  if (x % 2 == 0) List(x / 2) else List()

val numbers = List(2, 3, 4, 5, 6)

// Mit map:
val result1 = numbers.map(half)
println(result1)
// Ausgabe:  List(List(1), List(), List(2), List(), List(3))

// Mit flatMap:
val result2 = numbers.flatMap(half)
println(result2)
// Ausgabe: List(1, 2, 3)


List(List(1), List(), List(2), List(), List(3))
List(1, 2, 3)


defined function half
numbers: List[Int] = List(2, 3, 4, 5, 6)
result1: List[List[Int]] = List(List(1), List(), List(2), List(), List(3))
result2: List[Int] = List(1, 2, 3)

In [3]:
def half(x: Int): Option[Int] =
if (x % 2 == 0) Some(x / 2) else None
val numbers = List(2, 3, 4, 5, 6)
val result1 = numbers.map(half)

val result2 = numbers.flatMap(half)

defined function half
numbers: List[Int] = List(2, 3, 4, 5, 6)
result1: List[Option[Int]] = List(
  Some(value = 1),
  None,
  Some(value = 2),
  None,
  Some(value = 3)
)
result2: List[Int] = List(1, 2, 3)

## .fold(i)(...) und .reduce(...)

### Gemeinsamkeiten

- Beide Methoden führen eine Operation über alle Elemente einer Collection aus  
- Ziel ist es, die Collection auf einen einzelnen Wert zu reduzieren

### Unterschiede
- `reduce`: verwendet das erste Element als Startwert  
- `fold`: benötigt einen expliziten Startwert


In [2]:
val l1 = List(1, 2, 3, 4)

l1.reduce((acc,elem)=> acc+elem)
l1.fold("3")((acc,elem)=> acc.toString+elem.toString)

l1.fold(0)((acc,elem)=> acc+elem)

// Mit reduce:
l1.reduce((a, b) => a + b)   
l1.reduce(_ + _)             

// Mit fold:
l1.foldRight("")((elem, acc) => acc.toString + elem.toString)   

l1: List[Int] = List(1, 2, 3, 4)
res2_1: Int = 10
res2_2: scala.Int | java.lang.String = "31234"
res2_3: Int = 10
res2_4: Int = 10
res2_5: Int = 10
res2_6: String = "4321"

## `foldLeft` vs. `foldRight`

### 1. Iterationsrichtung

- `foldLeft`: arbeitet sich von links nach rechts durch die Collection
- `foldRight`: beginnt rechts und arbeitet sich nach links vor

### 2. Stellung des Initialwerts

- Bei `foldLeft` steht der Initialwert links in der Berechnung
- Bei `foldRight` steht der Initialwert rechts

In [1]:
val l2 = List(1, 2, 3, 4, 5, 7)

l2.foldLeft(0)(_ - _)
// ((((0 - 1) - 2) - 3) - 4) - 5 - 7 = -22

l2.foldRight(0)(_ - _)
// 1 - (2 - (3 - (4 - (5 - (7 - 0))))) = -4

val l3 = List("H", "e", "l", "l", "o")

l3.foldRight(" World!")  ((elem,acc) => elem + acc ) 

l2: List[Int] = List(1, 2, 3, 4, 5, 7)
res1_1: Int = -22
res1_2: Int = -4
l3: List[String] = List("H", "e", "l", "l", "o")
res1_4: String = "Hello World!"